In [1]:
from transformers import AutoModel,AutoTokenizer
from datasets import load_dataset

ds = load_dataset("tomaarsen/conllpp")

In [74]:
import torch
import numpy as np

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [3]:
ds["train"].features

{'id': Value(dtype='string', id=None),
 'document_id': Value(dtype='int32', id=None),
 'sentence_id': Value(dtype='int32', id=None),
 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'pos_tags': Sequence(feature=ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], id=None), length=-1, id=None),
 'chunk_tags': Sequence(feature=ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], id=None), length=-1, id=None),
 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC'

In [4]:
ds.set_format(type="pandas")

In [5]:
ds["train"][:]

,id,document_id,sentence_id,tokens,pos_tags,chunk_tags,ner_tags
0,0,1,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]"
2,2,1,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]"
3,3,1,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,1,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."
...,...,...,...,...,...,...,...
14036,14036,946,3,"[on, Friday, :]","[15, 22, 8]","[13, 11, 0]","[0, 0, 0]"
14037,14037,946,4,"[Division, two]","[21, 11]","[11, 12]","[0, 0]"
14038,14038,946,5,"[Plymouth, 2, Preston, 1]","[21, 11, 22, 11]","[11, 12, 12, 12]","[3, 0, 3, 0]"
14039,14039,946,6,"[Division, three]","[21, 11]","[11, 12]","[0, 0]"


In [6]:
classes = ds["train"].features["ner_tags"].feature.names

## conversion of ner_tags to labels

In [8]:
def converttolabels(batch):
    ner_tags_label = {"ner_tags_str": [ classes[idx] for idx in batch["ner_tags"]]}
    return ner_tags_label

In [9]:
ds.reset_format()

In [10]:
converttolabels(ds["train"][2])

{'ner_tags_str': ['B-LOC', 'O']}

In [11]:
ds_encoded = ds.map(converttolabels)

In [12]:
ds_encoded

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_str'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_str'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_str'],
        num_rows: 3453
    })
})

In [14]:
model_name = "distilbert/distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [15]:
tokenizer.is_fast

True

In [16]:
inputs = ds["train"][0]["tokens"]
out = tokenizer(inputs,is_split_into_words=True)

In [17]:
out

{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## the word "lamb" divided into two tokens so there is problem in alignment with the input tokens

In [18]:
out.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [19]:
out.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [20]:
classes

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

## alignment of ner_tags for a single sentence 

In [21]:
def alignment_to_input(labels,word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
    
        elif word_id is None:
            # in pytorch default padding value is -100
            new_labels.append( -100)
        else:
            label  = labels[word_id]
            if label % 2 == 1:
                label = label + 1
            new_labels.append(label)
    return new_labels

In [22]:
labels = ds["train"][0]["ner_tags"]
word_ids = out.word_ids()
word_ids,labels

([None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None], [3, 0, 7, 0, 0, 0, 7, 0, 0])

In [23]:
alignment_to_input(labels,word_ids)

[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]

### Tokenization and alignment of labels 

In [24]:
def tokens_align(tokens):
    tokenized_out = tokenizer(tokens["tokens"],truncation=True,is_split_into_words=True)
    all_labels = tokens["ner_tags"]
    new_labels = []
    for i, label in enumerate(all_labels):
        word_ids = tokenized_out.word_ids(i)
        new_labels.append(alignment_to_input(label,word_ids))
    tokenized_out["labels"] = new_labels

    return tokenized_out

In [25]:
batch = ds.map(tokens_align,batched= True,remove_columns=ds["train"].column_names)
batch

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [26]:
print(batch["train"][:2])

{'input_ids': [[101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], [101, 1943, 14428, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1]], 'labels': [[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100], [-100, 1, 2, -100]]}


## Padding of Dataset and labels(-100 for labels)

In [27]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer = tokenizer)

In [28]:
sample = data_collator([batch["train"][i] for i in range(2)])
sample

{'input_ids': tensor([[  101,  7270, 22961,  1528,  1840,  1106, 21423,  1418,  2495, 12913,
           119,   102],
        [  101,  1943, 14428,   102,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([[-100,    3,    0,    7,    0,    0,    0,    7,    0,    0,    0, -100],
        [-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100]])}

## Evaluation metrics

In [29]:
!pip install seqeval
!pip install evaluate

In [30]:
import evaluate
eval_metric = evaluate.load("seqeval")

In [31]:
label_names = ds["train"].features["ner_tags"].feature.names

In [32]:
labels = ds["train"][0]["ner_tags"]
true_labels = [label_names[i] for i in labels]

In [33]:
true_labels

['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']

In [34]:
predictions = true_labels.copy()
predictions[2] = "O"
predictions[6] = "O"

In [35]:
eval_metric.compute(predictions = [predictions],references=[true_labels])

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'MISC': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2},
 'ORG': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'overall_precision': 1.0,
 'overall_recall': 0.3333333333333333,
 'overall_f1': 0.5,
 'overall_accuracy': 0.7777777777777778}

In [77]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits,axis  = -1)
    true_labels = [[ label_names[l] for l in label if l != -100] for label in labels]
    pred_labels = [[ label_names[p] for p,l in zip(prediction,label) if l != -100] for prediction,label in zip(predictions,labels)]

    metric = eval_metric.compute(predictions = pred_labels,references = true_labels)

    return {"precision":metric["overall_precision"],
            "recall":metric["overall_recall"],
            "f1":metric["overall_f1"]}

In [62]:
token_clf.config.num_labels

9

In [64]:
from transformers import TrainingArguments,Trainer

training_args = TrainingArguments("../model",
                                 eval_strategy="epoch",
                                  learning_rate=1e-5,
                                 per_device_eval_batch_size=128,
                                 per_device_train_batch_size=128,
                                 weight_decay=0.01,
                                 save_strategy="epoch",num_train_epochs=5)

In [38]:
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [39]:
labels

[3, 0, 7, 0, 0, 0, 7, 0, 0]

In [40]:
label2id = {ch:i for i, ch in enumerate(label_names)}
id2label = dict(map(reversed,label2id.items()))
label2id,id2label

({'O': 0,
  'B-PER': 1,
  'I-PER': 2,
  'B-ORG': 3,
  'I-ORG': 4,
  'B-LOC': 5,
  'I-LOC': 6,
  'B-MISC': 7,
  'I-MISC': 8},
 {0: 'O',
  1: 'B-PER',
  2: 'I-PER',
  3: 'B-ORG',
  4: 'I-ORG',
  5: 'B-LOC',
  6: 'I-LOC',
  7: 'B-MISC',
  8: 'I-MISC'})

In [52]:
from transformers import AutoModelForTokenClassification
token_clf = AutoModelForTokenClassification.from_pretrained(model_name,id2label = id2label, label2id = label2id)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [44]:
token_clf

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [78]:
for params in token_clf.parameters():
    params.requires_grad = False

In [90]:
for params in token_clf.classifier.parameters():
    params.requires_grad = True

for params in token_clf.distilbert.transformer.layer[-6:].parameters():
    params.requires_grad = True

In [91]:
trainable = sum(params.numel() for params in token_clf.parameters() if params.requires_grad)
total = sum(params.numel() for params in token_clf.parameters())
trainable,total

(42534153, 65197833)

In [92]:
trainer = Trainer(token_clf,args = training_args,data_collator=data_collator,
                  train_dataset=batch["train"],
                 eval_dataset=batch["validation"],compute_metrics=compute_metrics)

In [93]:
trainer.train()

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 5.47 GiB is allocated by PyTorch, and 1.11 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [95]:
results = trainer.predict(batch["test"])
results.metrics

OutOfMemoryError: CUDA out of memory. Tried to allocate 198.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 5.76 GiB is allocated by PyTorch, and 1.01 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)